# Curva de precio horario a largo plazo · TFM Energía UCM

Una sola función, `curva(desde, hasta)`, que devuelve precio horario para **cualquier**
rango y dice de dónde sale cada día:

| origen | de dónde | cuándo se usa |
|---|---|---|
| `historico` | `spot_price`, el PMD publicado | hasta ayer |
| `modelo` | tabla `predictions`, el ensemble | los días que se hayan predicho |
| `simulado` | generado, con banda P10-P90 | más allá |

## Por qué los modelos de D+1 no sirven para esto

Los ocho modelos entrenados reciben **los 7 días previos observados** más las previsiones de
D+1. Para llegar a 2046 habría que realimentar sus propias predicciones 7.300 veces: el error
se compone y en pocos días la serie se aplana a la media. Y los exógenos de D+1 —demanda
prevista, eólica, gas, CO2— no existen para dentro de veinte años.

Un modelo de D+1 **explota la persistencia**. Una curva a largo plazo tiene que ignorarla.
No es la misma herramienta.

## Lo que sí se hace

La descomposición estándar del sector, en tres piezas deliberadamente separadas:

```
precio(dia, hora)  =  nivel(año) x factor_mes  +  forma(mes, tipo_dia, hora)  +  residuo
```

**El nivel no se predice: se aporta.** Sale de los futuros MIBEL —que cotizan a tres o cuatro
años— o de un escenario fundamental. Si no se pasa, el script usa la media de los últimos 12
meses en plano y **avisa de que es un marcador de posición, no una previsión**. Esa separación
es lo que hace defendible el resultado: la parte que se inventa está aislada y etiquetada.

**La forma sí sale de los datos**, y es la parte interesante.

**La banda** sale de remuestrear residuos en bloques de 24 horas.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "gold").is_dir())
sys.path.append(str(REPO / "scripts"))

# Recarga forzada: `curva_precios.py` esta en desarrollo, y Python cachea los modulos
# ya importados. Sin esto, añadir una funcion al script no llega al kernel y salta un
# ImportError que parece un error de nombre y es de cache.
import importlib, curva_precios
importlib.reload(curva_precios)
from curva_precios import (curva, deformacion, historico, perfil,
                           drivers, modelo_spread, por_anclas,
                           dias_molde, simular)

H = historico()
print(f"histórico: {H.dia.min():%Y-%m-%d} -> {H.dia.max():%Y-%m-%d} "
      f"· {H.dia.nunique():,} días · {len(H):,} horas")

## 1 · El perfil se está deformando

Este es el hallazgo que condiciona todo lo demás, y sale directo del histórico.

La tabla mide, para cada año, cuánto se desvía cada hora respecto a **la media de su propio
día** — así se aísla la forma del nivel.

In [ ]:
d = deformacion(H)
display(d)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].bar(d.index, d.nivel_medio, color="steelblue")
ax[0].set_title("Nivel medio anual (€/MWh)")
ax[0].set_ylabel("€/MWh"); ax[0].grid(alpha=.3, axis="y")

ax[1].plot(d.index, d.valle_12_15h, "o-", label="valle 12-15 h", color="darkorange")
ax[1].plot(d.index, d.pico_19_21h, "o-", label="pico 19-21 h", color="crimson")
ax[1].fill_between(d.index, d.valle_12_15h, d.pico_19_21h, alpha=.12, color="grey")
ax[1].axhline(0, color="black", lw=.8)
ax[1].set_title("Desviación sobre la media del día (€/MWh)")
ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f"el spread intradiario pasa de {d.spread.iloc[0]:.1f} a {d.spread.iloc[-1]:.1f} €/MWh "
      f"-> x{d.spread.iloc[-1]/d.spread.iloc[0]:.0f}")

**Dos cosas que conviene leer juntas.**

El nivel medio lleva plano desde 2024 —63, 65, 66 €/MWh— mientras el spread intradiario **se
ha duplicado**. Son fenómenos independientes: el nivel lo marca el gas y la demanda; el spread
lo marca cuánta solar entra al mediodía.

Y eso es exactamente lo que decide la rentabilidad de una batería. **No gana dinero porque el
precio suba, gana porque el precio de las 14:00 y el de las 20:00 se separen.** El capítulo de
almacenamiento se sostiene sobre esta gráfica, no sobre la de la izquierda.

Por eso el perfil se estima solo con los **últimos dos años**: promediar 2020 con 2026 daría
una forma que no existió nunca y que ya no va a volver.

## 2 · El perfil estimado, hora a hora

Lo que el simulador usa como forma: cuánto se desvía cada hora de la media de su día, por mes
y tipo de día.

In [ ]:
forma, fac_mes, res = perfil(H)
f = forma.reset_index()

fig, ax = plt.subplots(figsize=(10, 4.5))
for mes, col in [(1, "#4a6fa5"), (4, "#5ed69a"), (7, "#fb923c"), (10, "#a78bfa")]:
    s = f[(f.mes == mes) & (f.tipo == "laborable")].set_index("hora").rel
    ax.plot(s.index, s.values, "o-", ms=3, color=col,
            label=["ene", "abr", "jul", "oct"][[1, 4, 7, 10].index(mes)])
ax.axhline(0, color="black", lw=.8)
ax.set_xlabel("hora"); ax.set_ylabel("€/MWh sobre la media del día")
ax.set_title("Perfil intradiario por mes (días laborables, últimos 2 años)")
ax.set_xticks(range(0, 24, 2)); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

print("factor estacional por mes (multiplica al nivel anual):")
print(fac_mes.round(3).to_string())

El valle de mediodía es mucho más profundo en **julio** que en enero: más horas de sol y más
producción fotovoltaica. En invierno el perfil casi se aplana y el pico se desplaza.

Eso importa para la batería: **su margen no es constante a lo largo del año**, y un cálculo con
el spread medio anual sobreestima el invierno y subestima el verano.

## 3 · La curva a futuro, con banda

Aquí se pide el rango. El `nivel` es un **escenario que aportas tú** — de los futuros MIBEL o
de un modelo fundamental. Si lo dejas en `None`, el script avisa de que está usando un
marcador de posición.

In [ ]:
# El rango es libre: cambia estas dos fechas y todo lo demas se ajusta.
DESDE, HASTA = "2027-01-01", "2030-12-31"
ESCENARIOS = 400

# Nivel anual en €/MWh, POR ANCLAS. Se dan los años que se saben y el resto se interpola,
# que es lo unico que permite pedir veinte años sin escribir veinte numeros.
#   - los primeros se anclan a los futuros MIBEL
#   - el ultimo es una hipotesis propia, y hay que decirlo en la memoria
ANCLAS_NIVEL = {2027: 66, 2030: 60, 2040: 55, 2046: 52}

a0, a1 = int(DESDE[:4]), int(HASTA[:4])
NIVEL = por_anclas(ANCLAS_NIVEL, a0, a1)
print("nivel anual (€/MWh):", {k: round(v, 1) for k, v in NIVEL.items()})

c = curva(DESDE, HASTA, nivel=NIVEL, n=ESCENARIOS)
print()
display(c.groupby("origen").agg(dias=("dia", "nunique"), media=("p50", "mean")).round(2))

## 4 · Cómo se ve

In [ ]:
s = c[c.origen == "simulado"].copy()
s["mes"] = s.dia.dt.to_period("M").dt.to_timestamp()
m = s.groupby("mes")[["p10", "p50", "p90"]].mean()

fig, ax = plt.subplots(2, 1, figsize=(12, 8))

ax[0].fill_between(m.index, m.p10, m.p90, alpha=.2, color="steelblue", label="P10-P90")
ax[0].plot(m.index, m.p50, color="steelblue", lw=2, label="P50")
hm = H[H.dia > H.dia.max() - pd.DateOffset(years=2)].copy()
hm["mes"] = hm.dia.dt.to_period("M").dt.to_timestamp()
ax[0].plot(hm.groupby("mes").precio.mean(), color="black", lw=1.6, label="histórico")
ax[0].set_ylabel("€/MWh"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[0].set_title("Media mensual: histórico y curva simulada con banda")

sem = s[(s.dia >= "2028-06-05") & (s.dia <= "2028-06-11")]
x = range(len(sem))
ax[1].fill_between(x, sem.p10, sem.p90, alpha=.2, color="crimson")
ax[1].plot(x, sem.p50, color="crimson", lw=1.6)
ax[1].axhline(0, color="black", lw=.8)
ax[1].set_title("Una semana de junio de 2028, hora a hora")
ax[1].set_xlabel("horas"); ax[1].set_ylabel("€/MWh"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

neg = (s.p10 < 0).mean() * 100
print(f"horas con P10 negativo: {neg:.1f}%  ·  ancho medio de la banda: "
      f"{(s.p90 - s.p10).mean():.1f} €/MWh")

## 5 · Lo que esta curva NO es

Conviene decirlo en la memoria con la misma claridad con la que se presenta el resultado.

**El nivel es un supuesto, no una predicción.** Todo el largo plazo cuelga del escenario que
se introduce en `NIVEL`. Si ese escenario está mal, la curva está mal por mucho que la forma
y la banda sean correctas.

**La banda mide la variabilidad histórica, no la incertidumbre del escenario.** Los
percentiles salen de remuestrear residuos de los últimos dos años: recogen cómo de variable es
el precio *dado* un nivel, no la probabilidad de que el nivel sea otro. La incertidumbre real a
diez años es bastante mayor que esta banda.

**La forma se supone estable, y no lo es.** Se estima con los últimos dos años y se mantiene
fija hacia adelante. Pero la sección 1 demuestra justo lo contrario: el valle se hunde año a
año. Extrapolar esa deriva es la mejora más obvia y es lo primero que haría en una siguiente
versión.

**No hay modelo fundamental detrás.** Un despacho real —PyPSA, Antares, PLEXOS— calcularía el
precio a partir del parque instalado, la demanda y los combustibles. Aquí se toma el nivel como
dato y solo se modela la forma.

## 6 · Proyectar la forma en vez de congelarla

La sección 5 admitía la debilidad más gorda: la forma se estima con los últimos dos años y se
mantiene fija. Pero la sección 1 demuestra que **la forma es justo lo que más se mueve**.

Aquí se arregla. La idea: en vez de extrapolar el precio, se modela **de qué depende la
forma** y se evalúa esa función en escenarios futuros.

La variable candidata es la capacidad solar instalada, que sí tiene proyecciones creíbles —el
PNIEC fija 76 GW de fotovoltaica para 2030— mientras que el precio no las tiene.

In [ ]:
from curva_precios import drivers, modelo_spread

D = drivers()
pred, info = modelo_spread(D)
print(f"{info['meses']} meses · R² = {info['R2']}")
print(f"cada GW de solar abre el spread {info['pendiente_EUR_por_GW']:+.2f} €/MWh")
print(f"correlación con solar {info['corr_solar']}  ·  con eólica {info['corr_eolica']}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].scatter(D.solar_gw, D.spread, s=22, c=D.index.year, cmap="viridis")
gw = np.linspace(D.solar_gw.min(), 120, 50)
ax[0].plot(gw, [pred(g) for g in gw], color="crimson", lw=1.6, label="ajuste lineal")
for g, et in [(76, "PNIEC 2030")]:
    ax[0].axvline(g, ls="--", color="grey", lw=1)
    ax[0].annotate(et, (g, ax[0].get_ylim()[0]), rotation=90, fontsize=8,
                   va="bottom", ha="right", color="grey")
ax[0].set_xlabel("solar instalada (GW)"); ax[0].set_ylabel("spread intradiario (€/MWh)")
ax[0].set_title("El spread crece con el parque solar"); ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].plot(D.index.to_timestamp(), D.solar_gw, color="darkorange", label="solar")
ax[1].plot(D.index.to_timestamp(), D.eolica_gw, color="steelblue", label="eólica")
ax[1].set_ylabel("GW instalados"); ax[1].legend(); ax[1].grid(alpha=.3)
ax[1].set_title("Y las dos crecen a la vez — de ahí la advertencia de abajo")
plt.tight_layout(); plt.show()

**La advertencia, y es importante.** La eólica correlaciona 0,77 con el spread y la solar
0,73: prácticamente igual. Las dos crecen con el tiempo, así que **el tiempo confunde a las
dos** y esto no demuestra que sea la solar la que abre el spread.

Como modelo de primer orden para proyectar, sirve — el mecanismo físico es conocido y la
gráfica de la izquierda es consistente con él. Como afirmación causal, no. En la memoria hay
que presentarlo así: *el spread crece con el parque renovable, y se usa la solar como variable
de proyección porque es la que tiene objetivos publicados*.

## 7 · Veinte años en una celda

Aquí está todo junto: nivel por escenario, forma proyectada según la capacidad solar prevista,
y banda de percentiles. **175.320 horas.**

In [ ]:
# ─── LOS DOS ESCENARIOS QUE HAY QUE APORTAR ────────────────────────────────────
# Nada de esto sale de los datos: son los supuestos sobre los que descansa la curva.
ANO_INI, ANO_FIN = 2027, 2046          # <- pide el rango que quieras

# 1. Nivel anual en €/MWh, por anclas interpoladas
ANCLAS_NIVEL = {2027: 66, 2030: 60, 2035: 57, 2040: 55, 2046: 52}

# 2. Capacidad solar en GW. El PNIEC fija 76 GW para 2030; despues, hipotesis propia.
ANCLAS_SOLAR = {2027: 67, 2030: 76, 2035: 95, 2040: 110, 2046: 125}
# ───────────────────────────────────────────────────────────────────────────────

NIVEL = por_anclas(ANCLAS_NIVEL, ANO_INI, ANO_FIN)
SOLAR = por_anclas(ANCLAS_SOLAR, ANO_INI, ANO_FIN)

C20 = curva(f"{ANO_INI}-01-01", f"{ANO_FIN}-12-31",
            nivel=NIVEL, solar_gw=SOLAR, n=200, verbose=False)

an = C20.assign(y=C20.dia.dt.year).groupby("y").agg(
    p10=("p10", "mean"), p50=("p50", "mean"), p90=("p90", "mean"))
sp = C20.assign(y=C20.dia.dt.year, h=C20.hora)
sp["rel"] = sp.p50 - sp.groupby("dia").p50.transform("mean")
an["valle"] = sp[sp.h.between(12, 15)].groupby("y").rel.mean()
an["pico"] = sp[sp.h.between(19, 21)].groupby("y").rel.mean()
an["spread"] = an.pico - an.valle
an["solar_GW"] = pd.Series(SOLAR)
an["h_negativas_%"] = C20.assign(y=C20.dia.dt.year).groupby("y").p50.apply(
    lambda x: (x < 0).mean() * 100)

fig = plt.figure(figsize=(13, 8.5))
g = fig.add_gridspec(2, 2, height_ratios=[1.25, 1])

a0 = fig.add_subplot(g[0, :])
hh = H.groupby(H.dia.dt.year).precio.mean()
a0.plot(hh.index, hh.values, "o-", color="black", lw=1.8, label="histórico")
a0.fill_between(an.index, an.p10, an.p90, alpha=.2, color="steelblue", label="P10-P90")
a0.plot(an.index, an.p50, color="steelblue", lw=2, label="P50 simulado")
a0.axvline(2026.5, ls="--", color="grey", lw=1)
a0.set_ylabel("€/MWh"); a0.legend(); a0.grid(alpha=.3)
a0.set_title(f"Precio medio anual · histórico {hh.index.min()}-{hh.index.max()} "
             f"y curva {ANO_INI}-{ANO_FIN}")

a1 = fig.add_subplot(g[1, 0])
dd = deformacion(H)
a1.plot(dd.index, dd.spread, "o-", color="black", label="observado")
a1.plot(an.index, an.spread, color="crimson", lw=2, label="proyectado")
a1.set_ylabel("spread intradiario (€/MWh)"); a1.legend(); a1.grid(alpha=.3)
a1.set_title("El spread, que es lo que paga la batería")

a2 = fig.add_subplot(g[1, 1])
tercios = [ANO_INI, (ANO_INI + ANO_FIN) // 2, ANO_FIN]
for y, col in zip(tercios, ["#4a6fa5", "#fb923c", "#c0392b"]):
    q = sp[(sp.y == y) & (sp.dia.dt.month == 7)].groupby("h").rel.mean()
    a2.plot(q.index, q.values, "o-", ms=3, color=col, label=str(y))
a2.axhline(0, color="black", lw=.8)
a2.set_xlabel("hora"); a2.set_ylabel("€/MWh sobre la media del día")
a2.set_title("Perfil de julio, cómo se ahonda"); a2.legend(); a2.grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f"{len(C20):,} horas · {C20.dia.nunique():,} días · "
      f"{ANO_FIN - ANO_INI + 1} años completos\n")
display(an.round(1))

## 8 · La curva horaria

Dibujar las 175.320 horas seguidas da un manchón: veinte años en un eje no dejan ver una
sola hora. Así que va en cuatro imágenes separadas, cada una a su tamaño.

Los tres números de siempre en todas: **mínimo estimado (P10), curva propuesta (P50) y
máximo estimado (P90)**.

In [ ]:
# percentiles consolidados por hora del dia, sobre TODO el periodo
cons = C20.groupby("hora").agg(
    minimo=("p10", "mean"), estimacion=("p50", "mean"), maximo=("p90", "mean")).round(2)
cons["banda"] = (cons.maximo - cons.minimo).round(1)
hoy = H[H.dia > H.dia.max() - pd.DateOffset(years=1)].groupby("hora").precio.mean()

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.fill_between(cons.index, cons.minimo, cons.maximo, alpha=.22, color="steelblue",
                label="rango estimado P10 - P90")
ax.plot(cons.index, cons.minimo, lw=1, color="steelblue", alpha=.7)
ax.plot(cons.index, cons.maximo, lw=1, color="steelblue", alpha=.7)
ax.plot(cons.index, cons.estimacion, "o-", color="#0d47a1", lw=2.6, ms=5,
        label="curva propuesta (P50)")
ax.plot(hoy.index, hoy.values, "--", color="black", lw=1.6, label="últimos 12 meses reales")
ax.axhline(0, color="grey", lw=.9)
ax.set_xticks(range(24)); ax.set_xlabel("hora del día"); ax.set_ylabel("€/MWh")
ax.set_title(f"Curva horaria consolidada {ANO_INI}-{ANO_FIN} · media de {C20.dia.nunique():,} días",
             fontsize=13)
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

La curva propuesta frente a lo que pasa hoy: **el valle se hunde y el pico sube**. La banda es
ancha a media tarde —cuando el precio depende de si hay viento o no— y estrecha de madrugada.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.5))
anos = [ANO_INI, ANO_INI + (ANO_FIN - ANO_INI) // 3,
        ANO_INI + 2 * (ANO_FIN - ANO_INI) // 3, ANO_FIN]
cols = ["#90caf9", "#42a5f5", "#1565c0", "#0d1b4b"]
for y, col in zip(anos, cols):
    q = C20[C20.dia.dt.year == y].groupby("hora").p50.mean()
    ax.plot(q.index, q.values, "o-", ms=4, lw=2, color=col, label=str(y))
ax.plot(hoy.index, hoy.values, "--", color="crimson", lw=2, label="hoy (12 meses reales)")
ax.axhline(0, color="grey", lw=.9)
ax.set_xticks(range(24)); ax.set_xlabel("hora del día"); ax.set_ylabel("€/MWh")
ax.set_title("Cómo evoluciona el perfil: el valle de mediodía se ahonda", fontsize=13)
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5.5))
dia = C20.groupby("dia").agg(p10=("p10", "mean"), p50=("p50", "mean"), p90=("p90", "mean"))
ax.fill_between(dia.index, dia.p10, dia.p90, alpha=.18, color="steelblue", lw=0,
                label="rango P10 - P90")
ax.plot(dia.index, dia.p50, color="steelblue", lw=.35, alpha=.8)
ax.plot(dia.index, dia.p50.rolling(90, center=True).mean(), color="#0d47a1", lw=2,
        label="curva propuesta (media móvil 90 d)")
hd = H.groupby("dia").precio.mean()
ax.plot(hd.index, hd.rolling(90, center=True).mean(), color="black", lw=2, label="histórico")
ax.axhline(0, color="grey", lw=.9)
ax.set_ylabel("€/MWh")
ax.set_title(f"Periodo completo · media diaria · {C20.dia.nunique():,} días", fontsize=13)
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# una semana concreta, hora a hora, para ver el detalle que las otras esconden
SEMANA = f"{(ANO_INI + ANO_FIN) // 2}-07-06"
sem = C20[(C20.dia >= SEMANA) & (C20.dia < pd.Timestamp(SEMANA) + pd.Timedelta(days=7))]
ts = sem.dia + pd.to_timedelta(sem.hora, unit="h")

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(ts, sem.p10, sem.p90, alpha=.25, color="steelblue", label="rango P10 - P90")
ax.plot(ts, sem.p50, color="#0d47a1", lw=1.8, label="curva propuesta (P50)")
ax.axhline(0, color="grey", lw=.9)
for d in pd.date_range(SEMANA, periods=7):
    ax.axvline(d, color="grey", lw=.5, alpha=.5)
ax.set_ylabel("€/MWh")
ax.set_title(f"Detalle horario · semana del {pd.Timestamp(SEMANA):%d-%m-%Y}", fontsize=13)
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f"  CURVA HORARIA CONSOLIDADA {ANO_INI}-{ANO_FIN}  ({len(C20):,} horas)")
print(f"  {'hora':>5s} {'minimo':>9s} {'propuesta':>11s} {'maximo':>9s} {'banda':>8s}")
print("  " + "-" * 48)
for h, r in cons.iterrows():
    marca = "  <- valle" if h == cons.estimacion.idxmin() else (
            "  <- pico" if h == cons.estimacion.idxmax() else "")
    print(f"  {h:5d} {r.minimo:9.2f} {r.estimacion:11.2f} {r.maximo:9.2f} "
          f"{r.banda:8.1f}{marca}")
print("  " + "-" * 48)
print(f"  {'media':>5s} {cons.minimo.mean():9.2f} {cons.estimacion.mean():11.2f} "
      f"{cons.maximo.mean():9.2f} {cons.banda.mean():8.1f}")
print()
print(f"  spread de la curva consolidada: "
      f"{cons.estimacion.max() - cons.estimacion.min():.1f} EUR/MWh")
print(f"  el de los ultimos 12 meses reales: {hoy.max() - hoy.min():.1f}")

## 8c · La curva horaria impresa

Los tres números que se piden para cada hora: **mínimo estimado, curva propuesta y máximo
estimado**. P10, P50 y P90.

La `estimacion` es la curva a usar. El `minimo` y el `maximo` no son el peor y el mejor caso
posibles: son los percentiles 10 y 90, o sea que **una de cada cinco horas caerá fuera de esa
banda**. Es deliberado — una banda P0-P100 sería tan ancha que no diría nada.

In [ ]:
# ── cambia esta fecha para imprimir cualquier dia ────────────────────────────
DIA = "2030-07-15"
# ─────────────────────────────────────────────────────────────────────────────

T = C20.rename(columns={"p10": "minimo", "p50": "estimacion", "p90": "maximo"})
T = T[["dia", "hora", "minimo", "estimacion", "maximo"]].copy()
T["banda"] = (T.maximo - T.minimo).round(1)

uno = T[T.dia == DIA].drop(columns="dia").set_index("hora")
print(f"  CURVA HORARIA · {pd.Timestamp(DIA):%d-%m-%Y}")
print(f"  {'hora':>5s} {'minimo':>9s} {'estimacion':>12s} {'maximo':>9s} {'banda':>8s}")
print("  " + "-" * 50)
i_min, i_max = uno.estimacion.idxmin(), uno.estimacion.idxmax()
for h, r in uno.iterrows():
    marca = "  <- valle" if h == i_min else ("  <- pico" if h == i_max else "")
    print(f"  {h:5d} {r.minimo:9.2f} {r.estimacion:12.2f} {r.maximo:9.2f} "
          f"{r.banda:8.1f}{marca}")
print("  " + "-" * 50)
print(f"  {'media':>5s} {uno.minimo.mean():9.2f} {uno.estimacion.mean():12.2f} "
      f"{uno.maximo.mean():9.2f} {uno.banda.mean():8.1f}")
print()
print(f"  spread del dia (pico - valle sobre la estimacion): "
      f"{uno.estimacion.max() - uno.estimacion.min():.1f} EUR/MWh")

print()
print(f"  LA CURVA ENTERA · {len(T):,} horas")
with pd.option_context("display.max_rows", 40, "display.width", 100):
    display(T.set_index(["dia", "hora"]))

Para imprimir **todas** las horas de golpe en vez de las 40 que muestra el resumen:

```python
with pd.option_context("display.max_rows", None):
    display(T.set_index(["dia", "hora"]))
```

Son 175.320 filas, así que el navegador va a sufrir. Para trabajar con ellas de verdad está el
CSV de la sección siguiente.

## 8b · Validación: simular un año que ya conocemos

Una curva a veinte años no se puede validar contra el futuro. Lo que sí se puede es
**esconder un año conocido**: se construye el molde con datos anteriores a 2025, se simula
2025 dándole solo su nivel medio, y se compara con lo que realmente pasó.

Eso es lo único que convierte esta curva en algo defendible en lugar de una gráfica bonita.

In [ ]:
prev = H[H.dia < "2025-01-01"]
molde_bt = dias_molde(prev, anos=2)
_, fac_bt, _ = perfil(prev, anos=2)

bt = simular("2025-01-01", "2025-12-31", {2025: 65.3}, molde_bt, fac_bt,
             n=200, amplitud={2025: 1.0}, percentiles=(1, 10, 50, 90, 99))
j = bt.merge(H[H.dia.dt.year == 2025][["dia", "hora", "precio"]], on=["dia", "hora"])

q = [0, 5, 25, 50, 75, 95, 100]
tab = pd.DataFrame({"real": [np.percentile(j.precio, x) for x in q],
                    "simulado_p50": [np.percentile(j.p50, x) for x in q]},
                   index=[f"p{x}" for x in q]).round(1)
tab["dif"] = (tab.simulado_p50 - tab.real).round(1)
display(tab)

cob = ((j.precio >= j.p1) & (j.precio <= j.p99)).mean() * 100
cob80 = ((j.precio >= j.p10) & (j.precio <= j.p90)).mean() * 100
print(f"media    real {j.precio.mean():.1f}  ·  simulado {j.p50.mean():.1f}")
print(f"h < 0    real {(j.precio<0).mean()*100:.2f}%  ·  simulado {(j.p50<0).mean()*100:.2f}%")
print(f"cobertura P1-P99 {cob:.1f}%  (ideal 98)  ·  P10-P90 {cob80:.1f}%  (ideal 80)")

fig, ax = plt.subplots(figsize=(12, 4))
sem = j[(j.dia >= "2025-06-09") & (j.dia <= "2025-06-15")]
x = range(len(sem))
ax.fill_between(x, sem.p10, sem.p90, alpha=.25, color="steelblue", label="P10-P90")
ax.plot(x, sem.p50, color="steelblue", lw=1.6, label="P50 simulado")
ax.plot(x, sem.precio, color="black", lw=1.4, label="real")
ax.axhline(0, color="grey", lw=.8)
ax.set_title("Una semana de junio de 2025: simulado a ciegas contra lo que ocurrió")
ax.set_xlabel("horas"); ax.set_ylabel("€/MWh"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

**Cómo se lee.** La mediana, el mínimo y el porcentaje de horas negativas encajan bien: 70,1
frente a 70,0, y 5,8 % de horas negativas frente a 6,3 %. La distribución del precio está bien
reproducida.

La cobertura de la banda P1-P99 sale en el 92 % cuando debería rondar el 98: **la banda es algo
más estrecha de lo que debiera**. Hay que decirlo — la incertidumbre real es mayor que la
dibujada.

**Y lo más importante para el capítulo de baterías:** fíjate en que el P50 es más suave que la
línea negra. Es inevitable —una mediana entre 200 escenarios promedia los extremos— pero
significa que **el P50 subestima el spread**. Si calculas el ingreso de una batería sobre la
curva P50, te saldrá menos de lo que ganaría en realidad.

Para el capítulo 7 hay que operar la batería **sobre escenarios individuales** y luego
promediar el ingreso, no operarla sobre el promedio de los escenarios. No es lo mismo, y la
diferencia va en la dirección de infravalorar el negocio.

## 9 · Exportar la curva completa

In [ ]:
# ─── la curva completa a disco ────────────────────────────────────────────────
# El nombre lleva el motor: en la sección 10 hay otra curva y confundirlas
# sería fácil y caro.
SALIDA = REPO / "data" / "gold" / f"curva_{ANO_INI}_{ANO_FIN}_clasico.csv"

exp = C20.copy()
exp.insert(1, "fecha_hora", exp.dia + pd.to_timedelta(exp.hora, unit="h"))
exp = exp.drop(columns=["dia"])
exp.to_csv(SALIDA, index=False, float_format="%.2f")

print(f"{SALIDA.name}  ·  {len(exp):,} filas  ·  {SALIDA.stat().st_size/2**20:.1f} MB")
print(f"columnas: {list(exp.columns)}\n")
print(exp.head(3).to_string(index=False))
print("   ...")
print(exp.tail(3).to_string(index=False))

# y el resumen anual aparte, que es lo que se pega en la memoria
RES = REPO / "data" / "gold" / f"curva_{ANO_INI}_{ANO_FIN}_clasico_anual.csv"
an.round(2).to_csv(RES)
print(f"\nresumen anual en {RES.name}")

**Lo que se ve en la gráfica de abajo a la derecha** es el resultado que importa para el
capítulo de almacenamiento: el valle de julio se hunde año tras año conforme entra solar, y el
pico de la tarde se separa. Esa distancia es literalmente el margen bruto por ciclo de una
batería.

Y conviene mirar la de arriba junto a la de la izquierda: **el precio medio baja mientras el
spread sube.** Un análisis que solo mirara el nivel concluiría que el negocio empeora. Es al
revés.

---

Las cuatro limitaciones de la sección 5 siguen vigentes salvo la tercera, que es la que acaba
de arreglarse: la forma ya no se congela. Pero se proyecta con un ajuste lineal de R² 0,53
sobre variables que el tiempo confunde, así que **la incertidumbre real de esta curva es
bastante mayor que la banda que dibuja**. La banda recoge la variabilidad del precio dado un
escenario; no recoge que el escenario pueda estar equivocado.

---

# 10 · Curva fundamental: la demanda residual como motor

Todo lo anterior parte de un **nivel de precio que se aporta** y deforma la forma histórica
con un factor que sale de los GW solares. La sección 5 ya admitía los agujeros; esta sección
los cierra cambiando el motor, no parcheándolo.

## Por qué el conductor estaba mal elegido

Medido sobre la matriz, dentro de cada año para que el gas no confunda:

| año | corr(precio, **demanda residual**) | corr(precio, solar instalada) |
|---|---|---|
| 2023 | **+0,810** | −0,310 |
| 2024 | **+0,755** | −0,383 |
| 2025 | **+0,838** | −0,533 |
| 2026 | **+0,778** | −0,516 |

Y si se quita la residual de un ajuste `precio ~ f(residual, gas, CO₂, hora)`, el R² fuera de
muestra cae a **−0,045**: peor que predecir la media. Es ella quien hace todo el trabajo.

## Y por qué la residual sí se puede proyectar

Porque se descompone en piezas con objetivo publicado más meteorología:

```
residual(h,d,año) = demanda(año) × perfil(h,d)
                  − solar_GW(año)  × rendimiento_solar  × radiación(h,d)
                  − eólica_GW(año) × rendimiento_eólico × viento(h,d)³
```

No añade supuestos: los GW ya se los pedía la sección 7. Cambia el canal por el que entran.

## La trampa que hay que esquivar

El factor de carga **medido** no vale como entrada. Con el denominador correcto —solo solar de
red, porque el autoconsumo no vierte a `ree_gsolar_mw`— y comparando enero-agosto de todos los
años, cae de 0,194 en 2020 a 0,139 en 2025 mientras la capacidad se multiplica por cinco.

Esa caída es **endógena**: cuando el precio se va a cero las plantas vierten y dejan de
generar, así que el factor de carga medido ya lleva dentro el efecto del precio que queremos
predecir. Usarlo contaría el recorte dos veces.

Por eso aquí se usa **radiación y viento**, que son exógenos de verdad. Y el vertido deja de
ser un supuesto: cuando la residual se hunde, la curva de oferta devuelve cero, y ese cero
*es* el vertido.

In [ ]:
import curva_fundamental as cfun
importlib.reload(cfun)

P = cfun.panel()
potencial, ir = cfun.rendimientos(P)
D = cfun.con_residual(P, potencial)
precio_of, ic = cfun.curva_oferta(D)

print(f"rendimiento ajustado con {ir['horas_limpias']:,} horas de {ir['de']:,} "
      f"(las de precio > {cfun.PRECIO_LIMPIO:g} €/MWh, sin vertido)")
print(f"   solar   η = {ir['eta_solar']}   R² = {ir['R2_solar']}")
print(f"   eólica  η = {ir['eta_eolica']}   R² = {ir['R2_eolica']}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))

a = ax[0]
a.plot(ic["centro"], ic["k"], "o-", ms=3, color="crimson", lw=2)
a.set_xlabel("demanda residual (MW)"); a.set_ylabel("k = precio / gas", color="crimson")
a.tick_params(axis="y", labelcolor="crimson")
a.set_title("La curva de oferta, dibujada por los datos")
a.grid(alpha=.3)
b = a.twinx()
b.plot(ic["centro"], ic["p0"], "s--", ms=3, color="steelblue", lw=1.6)
b.set_ylabel("P(precio ≤ 0)", color="steelblue")
b.tick_params(axis="y", labelcolor="steelblue")

sub = D.sample(6000, random_state=42)
sc = ax[1].scatter(sub.residual, sub.precio, s=5, c=sub.ano, cmap="viridis", alpha=.5)
ax[1].axhline(0, color="black", lw=.9)
ax[1].set_xlabel("demanda residual (MW)"); ax[1].set_ylabel("€/MWh")
ax[1].set_title("Precio horario contra demanda residual")
plt.colorbar(sc, ax=ax[1], label="año"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f"\nk va de {ic['k_min']} a {ic['k_max']} · P(≤0) de {ic['p0_max']} a {ic['p0_min']}")
print(f"de los precios ≤ 0, un {ic['frac_cero_exacto']:.0%} son CERO EXACTO")

**Cómo se lee la izquierda.** `k` es el *heat rate* implícito de la planta marginal: cuánto
cuesta la hora por cada euro de gas. Cerca de 0 cuando margina renovable o nuclear, cerca de
3,5 cuando margina la térmica cara. Es monótona porque más demanda residual nunca puede
abaratar la hora — eso no se impone, sale así.

Dividir por el gas **antes** de ajustar es lo que hace que la curva extrapole a veinte años:
lo que se memoriza es la forma del *merit order*, no el nivel de precios de 2020-2024.

La línea azul es la novedad que arregla el defecto 2: la probabilidad de que la hora se case
a cero o por debajo, que pasa del 0 % con residual alta al 62 % cuando se hunde. Y de esos
precios ≤ 0, cuatro de cada diez son **cero exacto** — una masa puntual que ninguna
distribución continua puede producir.

In [ ]:
# ── las dos curvas, escondiendo 2025 ─────────────────────────────────────────
ANO_BT, N_BT = 2025, 200

# 1. la fundamental: NO recibe el precio, solo gas, demanda y capacidad reales de 2025
r = cfun.backtest(ANO_BT, n=N_BT, verbose=False)

# 2. la de la seccion 3: recibe el NIVEL REAL de 2025 como dato de entrada
prev = H[H.dia < f"{ANO_BT}-01-01"]
nivel_real = H[H.dia.dt.year == ANO_BT].precio.mean()
_, fac_bt, _ = perfil(prev, anos=2)
bt2 = simular(f"{ANO_BT}-01-01", f"{ANO_BT}-12-31", {ANO_BT: nivel_real},
              dias_molde(prev, anos=2), fac_bt, n=N_BT, amplitud={ANO_BT: 1.0},
              percentiles=(1, 10, 50, 90, 99))
j2 = bt2.merge(H[H.dia.dt.year == ANO_BT][["dia", "hora", "precio"]], on=["dia", "hora"])

def _spread(f, col):
    rel = f[col] - f.groupby("dia")[col].transform("mean")
    return rel[f.hora.between(19, 21)].mean() - rel[f.hora.between(12, 15)].mean()

print(f"  BACKTEST {ANO_BT} · las dos ajustadas solo con años anteriores\n")
print(f"  {'':22s} {'real':>9s} {'fundamental':>13s} {'sec. 3':>9s}")
print("  " + "-" * 58)
print(f"  {'nivel medio':22s} {r['media_real']:9.2f} {r['media_sim']:13.2f} "
      f"{j2.p50.mean():9.2f}")
print(f"  {'  ¿se lo damos?':22s} {'':>9s} {'NO, lo deduce':>13s} {'SÍ':>9s}")
print(f"  {'horas ≤ 0  %':22s} {r['neg_real']:9.2f} {r['neg_sim']:13.2f} "
      f"{(j2.p50 <= 0).mean()*100:9.2f}")
print(f"  {'horas = 0 exacto %':22s} {r['cero_real']:9.2f} {r['cero_sim']:13.2f} "
      f"{(j2.p50 == 0).mean()*100:9.2f}")
print(f"  {'spread pico-valle':22s} {r['spread_real']:9.2f} {r['spread_sim']:13.2f} "
      f"{_spread(j2, 'p50'):9.2f}")
print("  " + "-" * 58)
print(f"  cobertura P10-P90   fundamental {r['cob_80']:.1f}%   sec.3 "
      f"{((j2.precio >= j2.p10) & (j2.precio <= j2.p90)).mean()*100:.1f}%   (ideal 80)")
print(f"  cobertura P1-P99    fundamental {r['cob_98']:.1f}%   sec.3 "
      f"{((j2.precio >= j2.p1) & (j2.precio <= j2.p99)).mean()*100:.1f}%   (ideal 98)")

# la distribucion completa, que es donde se ve la masa en el suelo
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].hist(r["j"].precio, bins=60, range=(-20, 200), alpha=.55, color="black",
           density=True, label="real")
ax[0].hist(r["sims"].ravel(), bins=60, range=(-20, 200), alpha=.5, color="crimson",
           density=True, label="fundamental (escenarios)")
ax[0].axvline(0, color="steelblue", lw=1.2)
ax[0].set_xlabel("€/MWh"); ax[0].set_title(f"Distribución del precio en {ANO_BT}")
ax[0].legend(); ax[0].grid(alpha=.3)

sem = r["j"][(r["j"].dia >= f"{ANO_BT}-06-09") & (r["j"].dia <= f"{ANO_BT}-06-15")]
x = range(len(sem))
ax[1].fill_between(x, sem.p10, sem.p90, alpha=.25, color="crimson", label="P10-P90")
ax[1].plot(x, sem.precio, color="black", lw=1.4, label="real")
ax[1].axhline(0, color="grey", lw=.8)
ax[1].set_title("Una semana de junio, simulada a ciegas")
ax[1].set_xlabel("horas"); ax[1].set_ylabel("€/MWh"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

**La asimetría es el resultado.** La curva de la sección 3 recibe el nivel de 2025 como dato
de entrada: no puede equivocarse en el nivel porque se lo damos. La fundamental lo **deduce**
de gas, demanda y capacidad, y se queda a un 7 %.

Y reproduce el cero exacto, que es lo que ninguna distribución continua puede hacer. De ahí
sale casi toda la diferencia de spread.

**Una advertencia sobre la columna del P50**, porque es fácil caer: la mediana entre 200
escenarios da 0,00 % de horas a cero. Una hora solo sale cero en el P50 si más de la mitad de
los escenarios coinciden en cero. Cualquier afirmación sobre la *distribución* —cuántas horas
a cero, qué spread— hay que hacerla sobre los escenarios, con `simular(..., crudo=True)`.

Es exactamente el mismo motivo por el que la batería del capítulo 7 hay que operarla escenario
a escenario y luego promediar el ingreso.

In [ ]:
# ─── ESCENARIOS FÍSICOS: nada de esto es un precio ───────────────────────────
# El tramo simulado arranca el día siguiente al último precio publicado, no en enero del
# año que viene: los meses que quedan del año en curso son los primeros que hacen falta.
ULT_REAL = H.dia.max()
SIM_DESDE = ULT_REAL + pd.Timedelta(days=1)
A_SIM = SIM_DESDE.year                     # año en curso
ANO_FIN = 2046                             # <- el horizonte que se pide
print(f"último precio publicado: {ULT_REAL:%Y-%m-%d}")
print(f"tramo simulado: {SIM_DESDE:%Y-%m-%d} -> {ANO_FIN}-12-31 "
      f"({ANO_FIN - A_SIM + 1} años de calendario)")

# Las anclas arrancan en el valor OBSERVADO del año en curso: así la curva empalma con la
# realidad y no con un número inventado.
P = cfun.panel()
obs = P[P.ano == P.ano.max()]
GAS_HOY = float(obs.gas_mibgas.mean())
DEM_HOY = float(obs.demanda.mean())
SOL_HOY = float(obs.solar_gw.mean())
EOL_HOY = float(obs.eolica_gw.mean())
print(f"observado en {P.ano.max()}:  gas {GAS_HOY:.1f} €/MWh · demanda {DEM_HOY:,.0f} MW · "
      f"solar {SOL_HOY:.1f} GW · eólica {EOL_HOY:.1f} GW")

ANCLAS_SOLAR = {A_SIM: SOL_HOY, 2030: 76, 2035: 95, 2040: 110, ANO_FIN: 125}

ESC = dict(
    # el gas cede despacio; es la hipótesis más frágil de las cuatro, y la sección
    # siguiente mide cuánto depende de ella la conclusión sobre el spread
    gas=por_anclas({A_SIM: GAS_HOY, 2035: GAS_HOY * .82, ANO_FIN: GAS_HOY * .74},
                   A_SIM, ANO_FIN),
    # electrificación: +1 % anual acumulado
    demanda=por_anclas({A_SIM: DEM_HOY, ANO_FIN: DEM_HOY * 1.01 ** (ANO_FIN - A_SIM)},
                       A_SIM, ANO_FIN),
    solar_gw=por_anclas(ANCLAS_SOLAR, A_SIM, ANO_FIN),
    eolica_gw=por_anclas({A_SIM: EOL_HOY, 2030: 43, 2040: 55, ANO_FIN: 62},
                         A_SIM, ANO_FIN))

potencial, ir2 = cfun.rendimientos(P)
D = cfun.con_residual(P, potencial)
precio_of, ic = cfun.curva_oferta(D)

CF, SIMS = cfun.simular(SIM_DESDE, f"{ANO_FIN}-12-31", **ESC,
                        potencial=potencial, precio=precio_of,
                        n=200, verbose=False, crudo=True)

y = CF.dia.dt.year.to_numpy()
anf = CF.assign(a=y).groupby("a")[["p10", "p50", "p90"]].mean()
anf["media"] = [SIMS[:, y == k].mean() for k in anf.index]
anf["h_cero_%"] = [(SIMS[:, y == k] <= 0).mean() * 100 for k in anf.index]
anf["solar_GW"] = pd.Series(ESC["solar_gw"])
rel = CF.p50 - CF.groupby("dia").p50.transform("mean")
anf["spread"] = (rel[CF.hora.between(19, 21)].groupby(y[CF.hora.between(19, 21)]).mean()
                 - rel[CF.hora.between(12, 15)].groupby(y[CF.hora.between(12, 15)]).mean())

# ── CONTRASTE: la curva de precio sobre la residual REALMENTE observada ─────
# Sin sortear tiempo. Si se sortea, el error del modelo de precio se mezcla con el de
# haber simulado un año típico cuando el real fue atípico -- y 2026 lo fue: con la
# capacidad de 2026 fija, su recurso renovable potencial de enero-agosto sale 15.892 MW
# contra una media de 14.500 en los seis años disponibles. +1.400 MW, casi todo eólica.
print()
for a_ in sorted(P.ano.unique())[-3:]:
    ct = cfun.contraste(int(a_), n=100, d=D)
    print(f"contraste {a_}:  media real {ct['media_real']:5.1f} · modelo "
          f"{ct['media_sim']:5.1f}   |   horas ≤0 real {ct['neg_real']:5.2f}% · modelo "
          f"{ct['neg_sim']:5.2f}%")

fig, ax = plt.subplots(2, 1, figsize=(13, 8.5))

hh = H.groupby(H.dia.dt.year).precio.mean()
ax[0].plot(hh.index, hh.values, "o-", color="black", lw=1.8, label="histórico")
ax[0].fill_between(anf.index, anf.p10, anf.p90, alpha=.18, color="crimson", lw=0,
                   label="fundamental P10-P90")
ax[0].plot(anf.index, anf.media, color="crimson", lw=2.2, label="fundamental (media)")
ax[0].plot(an.index, an.p50, color="steelblue", lw=2, ls="--",
           label="sección 7 (nivel aportado)")
ax[0].axvline(A_SIM - .5, ls="--", color="grey", lw=1)
ax[0].set_ylabel("€/MWh"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[0].set_title(f"Las dos curvas · fundamental desde {SIM_DESDE:%Y-%m-%d}")

ax[1].plot(anf.index, anf["h_cero_%"], "o-", color="crimson", lw=2, label="fundamental")
hz = H.assign(a=H.dia.dt.year).groupby("a").precio.apply(lambda x: (x <= 0).mean() * 100)
ax[1].plot(hz.index, hz.values, "o-", color="black", lw=1.8, label="observado")
ax[1].set_ylabel("horas a precio ≤ 0 (%)"); ax[1].legend(); ax[1].grid(alpha=.3)
ax[1].set_title("La canibalización, que la sección 7 no podía representar")
plt.tight_layout(); plt.show()

print(f"\n{len(CF):,} horas · {CF.dia.nunique():,} días simulados")
display(anf.round(2))

## 10b · Diagnosis del residuo, con la receta de Box-Jenkins

La clase de series temporales insiste en un paso que es fácil saltarse: **antes de dar por
buenos los intervalos hay que comprobar que el residuo es ruido blanco**. Si la ACF del
residuo no se corta, el modelo está incompleto y —lo que importa aquí— la banda está mal
calculada.

Aplicado a este modelo, el diagnóstico es demoledor y explica la cobertura corta.

In [ ]:
# residuo log del modelo, en orden cronológico
res = ic["resid"]

print(f"residuo log · {len(res):,} horas · sd {res.std():.3f}\n")
print(f"  {'retardo':>8s} {'ACF':>7s}")
lags = [1, 2, 3, 6, 12, 24, 48, 168, 336, 720]
acf = [float(np.corrcoef(res[:-L], res[L:])[0, 1]) for L in lags]
for L, a in zip(lags, acf):
    print(f"  {L:8d} {a:7.3f}   {'#' * int(abs(a) * 40)}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].stem(lags, acf, basefmt=" ")
ax[0].axhline(0, color="black", lw=.8)
ax[0].axhline(2 / np.sqrt(len(res)), ls="--", color="crimson", lw=1,
              label="banda de ruido blanco (±2/√n)")
ax[0].axhline(-2 / np.sqrt(len(res)), ls="--", color="crimson", lw=1)
ax[0].set_xscale("log"); ax[0].set_xlabel("retardo (horas, escala log)")
ax[0].set_ylabel("ACF"); ax[0].set_title("El residuo NO es ruido blanco")
ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

# la consecuencia: al agregar, un ruido i.i.d. se promedia y el real no
dr = D[(D.ano >= ic["ajustada_desde"]) & (D.precio > 0)].sort_values(["dia", "hora"])
dr = dr.iloc[:len(res)].assign(e=res)
esc = {"hora": 1, "día": 24, "semana": 168, "mes": 720}
real_sd = [dr.e.groupby(dr.dia.dt.to_period("D" if k == "día" else
           ("W" if k == "semana" else "M"))).mean().std() if k != "hora" else dr.e.std()
           for k in esc]
teor_sd = [res.std() / np.sqrt(v) for v in esc.values()]
x = np.arange(len(esc))
ax[1].bar(x - .2, real_sd, .4, label="dispersión real", color="crimson")
ax[1].bar(x + .2, teor_sd, .4, label="si fuera ruido blanco", color="steelblue")
ax[1].set_xticks(x); ax[1].set_xticklabels(esc.keys())
ax[1].set_yscale("log"); ax[1].set_ylabel("sd del residuo agregado (log)")
ax[1].set_title("Por eso la banda salía estrecha")
ax[1].legend(); ax[1].grid(alpha=.3, axis="y")
plt.tight_layout(); plt.show()

for k, r_, t_ in zip(esc, real_sd, teor_sd):
    print(f"  por {k:7s} real {r_:.3f}   ruido blanco {t_:.3f}   x{r_/t_:.1f}")

**Lo que dice el diagnóstico.** La ACF vale 0,80 a un retardo, 0,39 a 24 horas y sigue en
0,12 al mes. No se corta en ningún sitio: el residuo tiene memoria a todas las escalas.

Y la barra de la derecha traduce eso a lo que importa. Un ruido independiente se promedia como
1/√n al agregar; el real no. A escala mensual la dispersión real es **catorce veces** la que
daría un ruido blanco.

Por eso la banda anual de la primera versión era de ±1 €/MWh en una curva a veinte años —
absurda, y no por optimismo sino por una hipótesis de independencia que los datos rechazan.

**El arreglo, y por qué no es un AR.** El temario apunta a modelar el residuo con un ARMA, que
sería lo natural. Pero un AR(1) con φ = 0,80 decae a 0,80⁷²⁰ ≈ 0 en un mes, y la persistencia
mensual medida es 0,12: haría falta un orden altísimo o memoria larga. Remuestrear **bloques
contiguos de 30 días** conserva todas las escalas a la vez sin elegir orden, y es lo que ya
hace `curva_precios` para los suyos.

Medido sobre 200 escenarios de 2025, dispersión de la media agregada entre escenarios:

| ruido | sd anual | sd mensual | banda anual P10-P90 |
|---|---|---|---|
| blanco | 1,05 | 3,61 | 69,7 – 72,3 |
| **bloques** | **5,90** | **19,21** | **60,7 – 76,9** |

La variabilidad interanual atribuible al residuo, medida sobre el histórico, es ~4,1 €/MWh.
Los bloques quedan en el mismo orden; el ruido blanco, cuatro veces corto.

**Honestidad sobre lo que NO arregla:** la cobertura *horaria* no mejora (73,2 % contra 78,5 %
con ruido blanco). Son dos fallos distintos — la cobertura horaria depende de la distribución
marginal de cada hora, la banda agregada depende de la dependencia entre horas. Los bloques
arreglan la segunda. La primera sigue necesitando calibración conforme.

## 10d · La curva horaria año a año, los dos motores enfrentados

Aquí se ve de una vez lo que las tablas anuales esconden: **cómo evoluciona el perfil de 24
horas y, sobre todo, qué hace la banda de percentiles** conforme entra capacidad renovable.

Cuatro años repartidos por el periodo, y en cada uno las dos estimaciones con su P10-P90:

- **fundamental** (rojo) — demanda residual, curva de oferta, suelo en cero, ruido por bloques
- **clásico** (azul) — nivel aportado y forma deformada por GW solares
- **hoy** (negro discontinuo) — los últimos 12 meses reales, como referencia fija

In [ ]:
# años a comparar: tienen que existir en las DOS curvas. La fundamental arranca en el año
# en curso y la clásica en el siguiente, así que se cruzan primero.
_comun = sorted(set(CF.dia.dt.year) & set(C20.dia.dt.year))
ANOS_VER = [_comun[0], _comun[len(_comun) // 3], _comun[2 * len(_comun) // 3], _comun[-1]]

def _perfil(df, ano):
    """P10, P50 y P90 medios por hora del día, para un año."""
    q = df[df.dia.dt.year == ano]
    return q.groupby("hora")[["p10", "p50", "p90"]].mean()

fig, axs = plt.subplots(2, 2, figsize=(14, 8.5), sharex=True, sharey=True)
for ax, ano in zip(axs.ravel(), ANOS_VER):
    f_, c_ = _perfil(CF, ano), _perfil(C20, ano)
    ax.fill_between(f_.index, f_.p10, f_.p90, alpha=.22, color="crimson", lw=0,
                    label="fundamental P10-P90")
    ax.plot(f_.index, f_.p50, "o-", ms=3, color="crimson", lw=2, label="fundamental P50")
    ax.fill_between(c_.index, c_.p10, c_.p90, alpha=.15, color="steelblue", lw=0,
                    label="clásico P10-P90")
    ax.plot(c_.index, c_.p50, "s--", ms=3, color="steelblue", lw=1.8, label="clásico P50")
    ax.plot(hoy.index, hoy.values, ":", color="black", lw=1.8, label="hoy (12 m reales)")
    ax.axhline(0, color="grey", lw=.9)
    ax.set_title(f"{ano}", fontsize=12)
    ax.set_xticks(range(0, 24, 3)); ax.grid(alpha=.3)
for ax in axs[1]:
    ax.set_xlabel("hora del día")
for ax in axs[:, 0]:
    ax.set_ylabel("€/MWh")
axs[0, 0].legend(fontsize=8, loc="upper center", ncol=2)
fig.suptitle("Curva horaria por año · las dos estimaciones con su banda", fontsize=13)
plt.tight_layout(); plt.show()

**Lo que hay que mirar es la banda, no la línea.**

En el clásico la banda es de anchura casi constante a lo largo del día y de los años: sale de
remuestrear residuos y no sabe nada de en qué hora está. En el fundamental la banda **se
estrecha por abajo al mediodía** conforme pasan los años — porque el P10 topa con el cero y no
puede seguir bajando — mientras sigue abierta en el pico de la tarde, donde el precio depende
de si hay viento.

Esa asimetría no es un adorno: es el suelo del mercado entrando en el percentil bajo. Es lo que
el clásico no puede representar y lo que decide cuántas horas baratas tendrá una batería.

In [ ]:
# ── cómo se comporta la banda: anchura por hora y por año ───────────────────
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))

cols = plt.cm.plasma(np.linspace(.15, .85, len(ANOS_VER)))
for ano, col in zip(ANOS_VER, cols):
    f_ = _perfil(CF, ano)
    ax[0].plot(f_.index, f_.p90 - f_.p10, "o-", ms=3, color=col, label=f"{ano} fund.")
    c_ = _perfil(C20, ano)
    ax[0].plot(c_.index, c_.p90 - c_.p10, "--", lw=1.2, color=col, alpha=.6)
ax[0].set_xticks(range(0, 24, 3)); ax[0].set_xlabel("hora del día")
ax[0].set_ylabel("anchura P10-P90 (€/MWh)")
ax[0].set_title("Anchura de la banda (continua: fundamental · discontinua: clásico)")
ax[0].legend(fontsize=8, ncol=2); ax[0].grid(alpha=.3)

# la asimetría: cuánto cae el P10 bajo el P50 frente a cuánto sube el P90
for df, nom, col in [(CF, "fundamental", "crimson"), (C20, "clásico", "steelblue")]:
    g = df.assign(a=df.dia.dt.year).groupby("a")[["p10", "p50", "p90"]].mean()
    ax[1].plot(g.index, g.p50 - g.p10, "-", color=col, lw=2, label=f"{nom}: P50 − P10")
    ax[1].plot(g.index, g.p90 - g.p50, "--", color=col, lw=2, label=f"{nom}: P90 − P50")
ax[1].set_xlabel("año"); ax[1].set_ylabel("€/MWh")
ax[1].set_title("Asimetría de la banda: la cola baja topa con el suelo")
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

# ── la tabla, que es lo que se pega en la memoria ───────────────────────────
print(f"  CURVA HORARIA POR AÑO · P10 / P50 / P90 · fundamental frente a clásico\n")
print(f"  {'h':>3s} " + "".join(
    f"| {a}  fund.        clás.       " for a in ANOS_VER[:2]))
print("  " + "-" * 76)
for h in range(24):
    fila = f"  {h:3d} "
    for a in ANOS_VER[:2]:
        f_, c_ = _perfil(CF, a).loc[h], _perfil(C20, a).loc[h]
        fila += (f"| {f_.p10:5.0f}{f_.p50:6.0f}{f_.p90:6.0f}  "
                 f"{c_.p10:5.0f}{c_.p50:6.0f}{c_.p90:6.0f} ")
    print(fila)
print("  " + "-" * 76)

resumen = pd.DataFrame({
    a: {"fund. banda media": (_perfil(CF, a).p90 - _perfil(CF, a).p10).mean(),
        "fund. banda al valle": (_perfil(CF, a).p90 - _perfil(CF, a).p10).loc[12:15].mean(),
        "fund. banda al pico": (_perfil(CF, a).p90 - _perfil(CF, a).p10).loc[19:21].mean(),
        "clás. banda media": (_perfil(C20, a).p90 - _perfil(C20, a).p10).mean(),
        "fund. spread P50": _perfil(CF, a).p50.max() - _perfil(CF, a).p50.min(),
        "clás. spread P50": _perfil(C20, a).p50.max() - _perfil(C20, a).p50.min()}
    for a in ANOS_VER}).round(1)
print()
display(resumen)

**Los dos números que resumen la comparación** están en la tabla: la anchura media de la
banda y el spread del P50.

El clásico mantiene una banda de anchura estable y un spread que crece; el fundamental
estrecha la banda en el valle —donde el cero hace de tope— y su spread satura. Son dos lecturas
distintas del mismo futuro y conviene presentar las dos.

Y hay una lectura para el capítulo de baterías que solo se ve aquí: **el P10 del fundamental se
pega al cero durante cada vez más horas del día**. Una batería no cobra el spread del P50: carga
en las horas del percentil bajo y descarga en el alto. Si el P10 se aplana en cero durante seis
horas en vez de dos, el coste de carga baja aunque el spread del P50 no se mueva.

**Y una asimetría que va en contra del fundamental, dicha aquí.** Mira el P10 de mediodía en
la tabla: el clásico llega a −15 €/MWh en 2033, el fundamental se queda en −1. El fundamental
reproduce bien la **masa en cero** pero es conservador en *cuánto* baja de ahí, porque su cola
negativa se remuestrea de un histórico donde los precios muy negativos todavía son raros.

O sea que las dos curvas se equivocan en direcciones opuestas también aquí: el clásico baja
sin tope porque no tiene suelo, el fundamental se frena en el suelo porque apenas ha visto lo
que hay debajo. Con más años de precios negativos —que van a llegar— esa cola se irá llenando
sola.

## Dónde discrepan las dos curvas, y de qué depende

La columna `spread` del fundamental **baja** con el escenario de arriba: de 104,7 en 2027 a
91,7 en 2046. La sección 7 la daba subiendo hasta 142.

Antes de sacar conclusiones hay que preguntarse de qué depende ese resultado, porque el
escenario de gas que hemos puesto es una hipótesis, no un dato. La celda siguiente lo mide.

In [ ]:
# ── ¿de qué depende la conclusión sobre el spread? ──────────────────────────
# El escenario de gas es la hipótesis más frágil de las cuatro. Si la conclusión se da la
# vuelta al cambiarlo, hay que presentarla condicionada y no como un hallazgo.
def _spread_nivel(gas_esc):
    c = cfun.simular(f"{ANO_INI}-01-01", f"{ANO_FIN}-12-31", gas=gas_esc,
                     demanda=ESC["demanda"], solar_gw=ESC["solar_gw"],
                     eolica_gw=ESC["eolica_gw"], potencial=potencial, precio=precio_of,
                     n=60, verbose=False)
    y = c.dia.dt.year.to_numpy()
    rel = c.p50 - c.groupby("dia").p50.transform("mean")
    sp = (rel[c.hora.between(19, 21)].groupby(y[c.hora.between(19, 21)]).mean()
          - rel[c.hora.between(12, 15)].groupby(y[c.hora.between(12, 15)]).mean())
    return sp, c.groupby(y).p50.mean()

print(f"  {'escenario de gas':30s} {'spread ' + str(ANO_INI):>12s} {str(ANO_FIN):>8s} "
      f"{'cambio':>9s} {'nivel ' + str(ANO_FIN):>11s}")
print("  " + "-" * 76)
sens = {}
for nom, g in [("baja (el de arriba)", ESC["gas"]),
               ("PLANO en el de hoy", {a: GAS_HOY for a in range(ANO_INI, ANO_FIN + 1)}),
               ("SUBE un 40 %", por_anclas({ANO_INI: GAS_HOY, ANO_FIN: GAS_HOY * 1.4},
                                           ANO_INI, ANO_FIN))]:
    sp, niv = _spread_nivel(g)
    sens[nom] = sp
    print(f"  {nom:30s} {sp.iloc[0]:12.1f} {sp.iloc[-1]:8.1f} "
          f"{sp.iloc[-1] - sp.iloc[0]:+9.1f} {niv.iloc[-1]:11.1f}")
# el clásico, con su propio escenario, para tener la referencia
spc = an.copy()
rel_c = C20.p50 - C20.groupby("dia").p50.transform("mean")
yc = C20.dia.dt.year.to_numpy()
sp_c = (rel_c[C20.hora.between(19, 21)].groupby(yc[C20.hora.between(19, 21)]).mean()
        - rel_c[C20.hora.between(12, 15)].groupby(yc[C20.hora.between(12, 15)]).mean())
print("  " + "-" * 76)
print(f"  {'CLÁSICO (nivel aportado)':30s} {sp_c.iloc[0]:12.1f} {sp_c.iloc[-1]:8.1f} "
      f"{sp_c.iloc[-1] - sp_c.iloc[0]:+9.1f}")

fig, ax = plt.subplots(figsize=(11, 4.6))
for (nom, sp), col in zip(sens.items(), ["#c0392b", "#e67e22", "#8e44ad"]):
    ax.plot(sp.index, sp.values, "-", lw=2, color=col, label=f"fundamental · gas {nom}")
ax.plot(sp_c.index, sp_c.values, "--", lw=2, color="steelblue", label="clásico")
dd = deformacion(H)
ax.plot(dd.index, dd.spread, "o-", color="black", lw=1.8, label="observado")
ax.set_ylabel("spread pico − valle (€/MWh)"); ax.set_xlabel("año")
ax.set_title("La conclusión sobre el spread depende del gas, no solo del suelo")
ax.legend(fontsize=9); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

**Lo que dice la medición, y corrige lo que parecía.** La caída del spread **no viene del
suelo en cero**: viene del escenario de gas. Con el gas plano el spread *sube* un poco, y con
el gas subiendo un 40 % sube mucho.

Lo que el suelo sí hace es amortiguar, y no es poco: con el mismo horizonte, el clásico crece
unos +45 €/MWh y el fundamental **+7 con el gas plano**. Convierte un crecimiento fuerte en uno
casi plano. Pero no lo invierte.

Así que la frase defendible es: *el suelo en cero frena el ensanchamiento del spread que la
extrapolación lineal proyecta; si además el gas cede, el spread se estrecha.* Las dos mitades
son necesarias y la segunda es una hipótesis.

---

# 11 · Qué curva se elige

**La fundamental es la curva del trabajo. La clásica se queda como contraste.**

Cuatro razones medidas, no de preferencia:

1. **Es la única que se puede validar.** El backtest de la clásica recibe el nivel real de 2025
   como entrada: no puede fallar en el nivel porque se lo damos. La fundamental lo deduce de
   gas, demanda y capacidad, y el contraste sobre la demanda residual observada da 65,4 contra
   63,0 en 2024, 65,4 contra 65,6 en 2025 y 72,0 contra 65,6 en 2026.

2. **Su conductor explica el precio y el de la clásica no.** Demanda residual: +0,78 a +0,84 de
   correlación con el precio horario dentro de cada año. Capacidad solar: −0,52, y confundida
   con la eólica por el tiempo, como admite la sección 6.

3. **Reproduce el cero.** 4,11 % de horas a cero exacto contra 2,79 % reales; la clásica, 0,00 %
   por construcción. Y las horas a cero son ya el 15,5 % del año.

4. **Su banda tiene la dispersión agregada correcta.** Tras el diagnóstico ACF y el remuestreo
   por bloques, la desviación anual entre escenarios es 5,90 €/MWh contra ~4,1 medidos. Con
   ruido independiente eran 1,05: una banda de ±1 €/MWh a veinte años.

**Para qué sirve entonces la clásica.** Para dos cosas concretas: como cota superior del
spread —crece sin tope porque su valle no tiene suelo, y eso acota por arriba el negocio de la
batería— y como control de que el nivel de la fundamental no se va, ya que la clásica se ancla
a un nivel exógeno de los futuros MIBEL.

**Lo que hay que escribir junto al resultado**, y no en un anexo:

- El nivel ya no se supone, pero **el gas sí**, y `k` multiplica directamente por `gas^0,713`.
  Se ha ganado en que el supuesto es sobre una magnitud que cotiza a plazo y se puede
  contrastar, no sobre el propio resultado.
- **Ninguna de las dos modela el almacenamiento.** Hoy hay 235 MW de batería en España, así que
  el efecto no se puede estimar de los datos — no hay variación. El PNIEC apunta a decenas de
  GW y el almacenamiento vive de arbitrar el spread: **el spread de 2046 es un techo en las dos
  curvas**, y el caso de negocio del capítulo 7 hereda ese sesgo al alza.
- La cobertura horaria sigue corta (73–78 % frente al 80 objetivo, 92 % frente al 98). La
  dependencia ya está bien; la distribución marginal de cada hora, no. Calibración conforme
  pendiente.
- El viento entra con R² 0,322 porque se usa el cubo de la media espacial. Se arregla con los
  tensores ECMWF a 0,25°, de los que faltan 705 días por descargar.

**El fichero.** `data/gold/curva_2027_2046_fundamental.csv`, 175.320 horas con P10, P50 y P90.
El `_clasico.csv` se conserva al lado, y el nombre de cada uno dice de qué motor viene.

## 10c · Exportar la curva fundamental

In [ ]:
# ─── la curva fundamental a disco, con el motor en el nombre ─────────────────
# El fichero de la sección 9 es el motor CLÁSICO. Que los dos convivan sin que se pueda
# saber cuál es cuál sería peor que no exportarlos.
SAL_F = REPO / "data" / "gold" / f"curva_{A_SIM}_{ANO_FIN}_fundamental.csv"

expf = CF.copy()
expf.insert(1, "fecha_hora", expf.dia + pd.to_timedelta(expf.hora, unit="h"))
expf = expf.drop(columns=["dia"])
expf.to_csv(SAL_F, index=False, float_format="%.2f")
print(f"{SAL_F.name}  ·  {len(expf):,} filas  ·  {SAL_F.stat().st_size/2**20:.1f} MB")

anf.round(2).to_csv(REPO / "data" / "gold" /
                    f"curva_{A_SIM}_{ANO_FIN}_fundamental_anual.csv")

print(f"\nlos dos motores desde el MISMO punto de entrada:")
print(f"    curva(desde, hasta, nivel=...)                      -> clásico (por defecto)")
print(f"    curva(desde, hasta, motor='fundamental', gas=...)   -> este")
print(f"\ny en disco:")
for f_ in sorted((REPO / "data" / "gold").glob("curva_*.csv")):
    print(f"    {f_.name:48s} {f_.stat().st_size/2**20:6.1f} MB")

# ── LA CURVA OPERATIVA COMPLETA: histórico + predicciones + simulado ────────
# `curva()` cose los tres orígenes. Es el fichero que se entrega: empieza en el primer
# día con dato y no se corta hasta 2046.
COMPLETA = curva(f"{H.dia.min():%Y-%m-%d}", f"{ANO_FIN}-12-31", motor="fundamental",
                 n=200, verbose=True, **ESC)
SAL_C = REPO / "data" / "gold" / f"curva_completa_{ANO_FIN}.csv"
expc = COMPLETA.copy()
expc.insert(1, "fecha_hora", expc.dia + pd.to_timedelta(expc.hora, unit="h"))
expc = expc.drop(columns=["dia"])
expc.to_csv(SAL_C, index=False, float_format="%.2f")
print(f"\n{SAL_C.name}  ·  {len(expc):,} filas  ·  {SAL_C.stat().st_size/2**20:.1f} MB")
display(COMPLETA.groupby("origen").agg(dias=("dia", "nunique"), desde=("dia", "min"),
                                       hasta=("dia", "max"), media=("p50", "mean")).round(2))

---

## Lo que sigue flojo, dicho aquí y no en el anexo

**El viento, R² 0,322.** El cubo de la media espacial no es la media de los cubos: promediar
el viento sobre toda la península y *después* elevarlo al cubo destruye justo la no
linealidad que hace útil la variable. La solución es la distribución espacial, o sea los
tensores ECMWF a 0,25° — de los que hoy faltan 705 días por descargar. Es la misma tarea.

**La cobertura sigue corta.** Seis años climáticos son seis sorteos: los años secos y sin
viento están infrarrepresentados y la banda sale estrecha por construcción. Calibrarla con
CQR es un parche honesto; treinta años de ERA5 sería el arreglo.

**Sin realimentación de almacenamiento.** Esta curva asume, sin decirlo hasta ahora, **cero
baterías para siempre**. Comprobado en la base: hoy España tiene 235 MW de batería híbrida,
así que la relación histórica está limpia — pero también por eso el efecto no se puede
estimar de los datos, no hay variación. Como el PNIEC apunta a decenas de GW y el
almacenamiento vive de arbitrar el spread, **el spread de 2046 es un techo, no una
estimación**, y el caso de negocio del capítulo 7 hereda ese sesgo al alza.

Va en dirección contraria al sesgo del P50, que lo subestima. Los dos existen y no sabemos
cuál gana. Decirlo así es más defendible que callar el segundo.

**El gas es la hipótesis frágil.** El nivel ya no se aporta como precio, pero sí como
combustible, y `k` multiplica directamente por él. Se ha ganado en que el supuesto es sobre
una magnitud que cotiza a plazo y se puede contrastar, no sobre el resultado.